# Day 7: FastAPI Advanced, Security and MongoDB
**Dependencies, middleware, JWT with scopes, OWASP API Top 10, MongoDB APIs, NoSQL injection and bandit**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Colab uses mongomock (in-memory MongoDB). VS Code can use a real MongoDB by setting environment variables.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx pyjwt bcrypt mongomock pymongo python-multipart bandit pip-audit
import os
os.environ["USE_MOCK"] = "1"

# ===== Option B: VS Code with real MongoDB (uncomment to use) =====
# pip install fastapi uvicorn httpx pyjwt bcrypt pymongo python-multipart bandit pip-audit   (in your terminal)
# import os
# os.environ["USE_MOCK"] = "0"
# os.environ["MONGO_URI"] = "mongodb://localhost:27017"   # or your Atlas URI

os.environ.setdefault("SECRET_KEY", "dev-only-change-me-use-32-plus-random-bytes")   # demo only: use a real secret from .env or Secrets
print("Mock mode:", os.environ["USE_MOCK"])

## 2. Dependency injection basics

In [ ]:
from fastapi import FastAPI, Depends, Query, HTTPException
from fastapi.testclient import TestClient

demo = FastAPI()

def pagination(skip: int = Query(0, ge=0), limit: int = Query(10, ge=1, le=50)):
    return {"skip": skip, "limit": limit}

def get_resource():
    print("  setup: open resource")
    yield {"name": "fake-db"}
    print("  cleanup: close resource")

@demo.get("/items")
def items(page=Depends(pagination), res=Depends(get_resource)):
    return {"page": page, "resource": res["name"]}

c = TestClient(demo)
print(c.get("/items", params={"limit": 5}).json())
print("Invalid limit:", c.get("/items", params={"limit": 500}).status_code)

## 3. Middleware and exception handlers

In [ ]:
import time, uuid
from fastapi.responses import JSONResponse

timed = FastAPI()

class TicketLocked(Exception):
    pass

@timed.exception_handler(TicketLocked)
def locked_handler(request, exc):
    return JSONResponse(status_code=409, content={"detail": "Ticket is closed"})

@timed.middleware("http")
async def timing(request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    ms = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-ms"] = f"{ms:.1f}"
    response.headers["X-Request-ID"] = str(uuid.uuid4())
    print(f"  {request.method} {request.url.path} -> {response.status_code} in {ms:.1f} ms")
    return response

@timed.get("/ping")
def ping():
    return {"pong": True}

@timed.patch("/tickets/{tid}")
def edit(tid: int):
    raise TicketLocked()

tc = TestClient(timed)
r = tc.get("/ping")
print({k: v for k, v in r.headers.items() if k.startswith("x-")})
print(tc.patch("/tickets/1").status_code, tc.patch("/tickets/1").json())

## 4. Password hashing

In [ ]:
import bcrypt

def hash_password(password: str) -> str:
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password: str, stored: str) -> bool:
    return bcrypt.checkpw(password.encode(), stored.encode())

h1, h2 = hash_password("secret123"), hash_password("secret123")
print("Same password, different hashes:", h1[:20], "...", h2[:20], "...")
print(verify_password("secret123", h1), verify_password("wrong", h1))

## 5. NEW (v4): Short-lived JWTs with scopes

In [ ]:
import jwt, datetime as dt
SECRET = os.environ["SECRET_KEY"]

def create_token(sub, role="customer", scopes=("tickets:read",), minutes=15):
    now = dt.datetime.now(dt.timezone.utc)
    payload = {"sub": sub, "role": role, "scope": " ".join(scopes), "iat": now,
               "exp": now + dt.timedelta(minutes=minutes)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

token = create_token("hana", scopes=["tickets:read", "tickets:write"])
print("Decoded:", jwt.decode(token, SECRET, algorithms=["HS256"]))

try:
    jwt.decode(create_token("hana", minutes=-1), SECRET, algorithms=["HS256"])
except jwt.ExpiredSignatureError:
    print("Expired token rejected")
try:
    jwt.decode(token, "a-different-key-that-is-32-bytes-long", algorithms=["HS256"])
except jwt.InvalidSignatureError:
    print("Forged token rejected")

## 6. The secure SmartDesk tickets API (FastAPI + JWT + MongoDB)
Ownership checks (OWASP API1), no role from the client (API3), capped page size (API4), admin-only route (API5), scopes on write routes.

In [ ]:
%%writefile secure_tickets.py
import datetime as dt, os, re
import bcrypt, jwt
from bson import ObjectId
from fastapi import Depends, FastAPI, HTTPException, Query
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, Field

SECRET = os.environ.get("SECRET_KEY", "dev-only-change-me-use-32-plus-random-bytes")
if os.environ.get("USE_MOCK", "1") == "1":
    import mongomock
    client = mongomock.MongoClient()
else:
    from pymongo import MongoClient
    client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
db = client["smartdesk"]
app = FastAPI(title="SmartDesk secure API")

ROLE_SCOPES = {"customer": ["tickets:read", "tickets:write"],
               "agent": ["tickets:read", "tickets:write", "analytics:read"],
               "admin": ["tickets:read", "tickets:write", "analytics:read", "admin"]}

def create_token(sub, role, minutes=15):
    now = dt.datetime.now(dt.timezone.utc)
    return jwt.encode({"sub": sub, "role": role, "scope": " ".join(ROLE_SCOPES[role]),
                       "iat": now, "exp": now + dt.timedelta(minutes=minutes)}, SECRET, algorithm="HS256")

oauth2 = OAuth2PasswordBearer(tokenUrl="token")

def current_user(token: str = Depends(oauth2)):
    try:
        data = jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token", headers={"WWW-Authenticate": "Bearer"})
    return {"username": data["sub"], "role": data["role"], "scopes": data["scope"].split()}

def require_scope(needed):
    def check(user=Depends(current_user)):
        if needed not in user["scopes"]:
            raise HTTPException(403, f"Missing scope {needed}")
        return user
    return check

class Register(BaseModel):                       # no role field: clients cannot make themselves admin (API3)
    username: str = Field(min_length=3, max_length=30, pattern=r"^[a-z0-9_]+$")
    password: str = Field(min_length=8, max_length=128)

class TicketIn(BaseModel):
    title: str = Field(min_length=3, max_length=120)
    priority: str = Field(pattern="^(low|medium|high)$")

def out(doc):
    return {"id": str(doc["_id"]), "title": doc["title"], "priority": doc["priority"],
            "status": doc["status"], "owner": doc["owner"]}

@app.post("/register", status_code=201, tags=["Auth"])
def register(user: Register):
    if db.users.find_one({"username": user.username}):
        raise HTTPException(409, "Username already taken")
    db.users.insert_one({"username": user.username, "role": "customer",
                         "password": bcrypt.hashpw(user.password.encode(), bcrypt.gensalt()).decode()})
    return {"username": user.username, "role": "customer"}

@app.post("/token", tags=["Auth"])
def login(form: OAuth2PasswordRequestForm = Depends()):
    u = db.users.find_one({"username": form.username})
    if not u or not bcrypt.checkpw(form.password.encode(), u["password"].encode()):
        raise HTTPException(401, "Wrong username or password")
    return {"access_token": create_token(u["username"], u["role"]), "token_type": "bearer"}

@app.post("/tickets", status_code=201, tags=["Tickets"])
def create_ticket(t: TicketIn, user=Depends(require_scope("tickets:write"))):
    doc = {**t.model_dump(), "status": "open", "owner": user["username"]}
    doc["_id"] = db.tickets.insert_one(doc).inserted_id
    return out(doc)

@app.get("/tickets", tags=["Tickets"])
def my_tickets(page: int = Query(1, ge=1), size: int = Query(10, ge=1, le=50),
               user=Depends(require_scope("tickets:read"))):
    flt = {} if user["role"] in ("agent", "admin") else {"owner": user["username"]}   # API1
    total = db.tickets.count_documents(flt)
    cur = db.tickets.find(flt).sort("_id", 1).skip((page - 1) * size).limit(size)
    return {"page": page, "size": size, "total": total, "items": [out(d) for d in cur]}

@app.delete("/tickets/{tid}", status_code=204, tags=["Tickets"])
def delete_ticket(tid: str, user=Depends(require_scope("tickets:write"))):
    if not ObjectId.is_valid(tid):
        raise HTTPException(404, "Ticket not found")
    res = db.tickets.delete_one({"_id": ObjectId(tid), "owner": user["username"]})   # API1
    if res.deleted_count == 0:
        raise HTTPException(404, "Ticket not found")

@app.get("/tickets/search", tags=["Tickets"])
def search(q: str = Query(min_length=2, max_length=40), user=Depends(require_scope("tickets:read"))):
    rx = {"$regex": re.escape(q), "$options": "i"}
    return [out(d) for d in db.tickets.find({"title": rx, "owner": user["username"]})]

@app.get("/analytics/by-priority", tags=["Analytics"])
def by_priority(user=Depends(require_scope("analytics:read"))):
    rows = db.tickets.aggregate([
        {"$match": {"status": {"$ne": "closed"}}},
        {"$group": {"_id": "$priority", "open": {"$sum": 1}}},
        {"$sort": {"open": -1}},
        {"$project": {"_id": 0, "priority": "$_id", "open": 1}},
    ])
    return list(rows)

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, secure_tickets
importlib.reload(secure_tickets)
api = TestClient(secure_tickets.app)

# ===== Option B: VS Code =====
# Terminal: uvicorn secure_tickets:app --reload
# Open /docs, use POST /register, then the green Authorize button to log in.

for u in ["hana", "ravi"]:
    print(api.post("/register", json={"username": u, "password": u + "-pass-123"}).json())
print("Try to self-register as admin:",
      api.post("/register", json={"username": "mallory", "password": "mallory-pass", "role": "admin"}).json())
secure_tickets.db.users.insert_one({"username": "agent1", "role": "agent",
    "password": secure_tickets.bcrypt.hashpw(b"agent-pass-123", secure_tickets.bcrypt.gensalt()).decode()})

def login(u, p):
    r = api.post("/token", data={"username": u, "password": p})
    return {"Authorization": f"Bearer {r.json()['access_token']}"} if r.status_code == 200 else r.status_code

print("Wrong password:", login("hana", "nope"))
hana, ravi, agent = login("hana", "hana-pass-123"), login("ravi", "ravi-pass-123"), login("agent1", "agent-pass-123")

In [ ]:
t1 = api.post("/tickets", json={"title": "No signal at home", "priority": "high"}, headers=hana).json()
api.post("/tickets", json={"title": "Charged twice", "priority": "medium"}, headers=hana)
api.post("/tickets", json={"title": "SIM not activated", "priority": "high"}, headers=ravi)

print("Hana sees:", [t["title"] for t in api.get("/tickets", headers=hana).json()["items"]])
print("Ravi sees:", [t["title"] for t in api.get("/tickets", headers=ravi).json()["items"]])
print("No token:", api.get("/tickets").status_code)
print("Ravi deletes Hana's ticket (API1):", api.delete(f"/tickets/{t1['id']}", headers=ravi).status_code)
print("Page size 1000 (API4):", api.get("/tickets", params={"size": 1000}, headers=hana).status_code)
print("Customer on analytics (scope):", api.get("/analytics/by-priority", headers=hana).status_code)
print("Agent analytics:", api.get("/analytics/by-priority", headers=agent).json())

### Testing with dependency overrides

In [ ]:
app = secure_tickets.app
app.dependency_overrides[secure_tickets.current_user] = lambda: {
    "username": "tester", "role": "admin", "scopes": secure_tickets.ROLE_SCOPES["admin"]}
print(TestClient(app).get("/analytics/by-priority").status_code)   # no token needed in tests
app.dependency_overrides.clear()
print(TestClient(app).get("/analytics/by-priority").status_code)   # back to normal: 401

## 7. NEW (v4): NoSQL injection, break it then fix it
The app below is **deliberately vulnerable**. It accepts raw JSON and passes it into queries. Only ever try this against your own local code.

In [ ]:
import mongomock
from fastapi import Body

weak_db = mongomock.MongoClient()["weak"]
weak_db.users.insert_one({"username": "admin", "password": "S3cret-not-hashed"})
weak_db.tickets.insert_many([{"title": "Router reset", "owner": "hana"},
                             {"title": "VIP outage at HQ", "owner": "admin"}])
weak = FastAPI()

@weak.post("/login")
def weak_login(body: dict = Body(...)):                       # raw dict: any shape accepted
    user = weak_db.users.find_one({"username": body.get("username"), "password": body.get("password")})
    return {"ok": bool(user)}

@weak.get("/search")
def weak_search(q: str):
    return [t["title"] for t in weak_db.tickets.find({"title": {"$regex": q}}, {"_id": 0})]

w = TestClient(weak)
print("Operator payload login:", w.post("/login", json={"username": "admin", "password": {"$ne": ""}}).json())
print("Regex payload search:", w.get("/search", params={"q": ".*"}).json())

In [ ]:
import re
from pydantic import BaseModel, Field

class Login(BaseModel):
    username: str = Field(min_length=3, max_length=30, pattern=r"^[a-z0-9_]+$")
    password: str = Field(min_length=8, max_length=128)

fixed = FastAPI()

@fixed.post("/login")
def fixed_login(body: Login):
    user = weak_db.users.find_one({"username": body.username})
    return {"ok": bool(user) and user["password"] == body.password}   # demo only: real code checks a hash

@fixed.get("/search")
def fixed_search(q: str = Query(min_length=2, max_length=40)):
    rx = {"$regex": re.escape(q), "$options": "i"}
    return [t["title"] for t in weak_db.tickets.find({"title": rx, "owner": "hana"}, {"_id": 0})]

f = TestClient(fixed)
print("Operator payload login:", f.post("/login", json={"username": "admin", "password": {"$ne": ""}}).status_code)
print("Regex payload search:", f.get("/search", params={"q": ".*"}).json())
print("Normal search:", f.get("/search", params={"q": "router"}).json())

### TODO 1: Add a third payload of your own (for example `{"$gt": ""}`) and show that the fixed app rejects it.

In [ ]:
# TODO: write your code here

## 8. NEW (v4): Security check for AI-generated code with bandit
This file imitates code an assistant might produce in a hurry. bandit reads the source and flags risky patterns.

In [ ]:
%%writefile ai_generated_helpers.py
import hashlib
import subprocess

SECRET_KEY = "super-secret-123"


def run_report(name):
    return subprocess.run(f"python make_report.py {name}", shell=True)


def fingerprint(password):
    return hashlib.md5(password.encode()).hexdigest()


def calculate(expr):
    return eval(expr)

In [ ]:
import json, subprocess, sys
res = subprocess.run([sys.executable, "-m", "bandit", "-q", "-f", "json", "ai_generated_helpers.py"],
                     capture_output=True, text=True)
findings = json.loads(res.stdout)["results"]
for f_ in findings:
    print(f"{f_['test_id']}  {f_['issue_severity']:<6} line {f_['line_number']:<3} {f_['issue_text'][:70]}")

## 9. NEW (v4): Dependency check with pip-audit
pip-audit checks pinned packages against known vulnerability data. It needs internet, so this cell is optional in the offline check. Old pins such as `PyYAML==5.3.1` are reported with an advisory id and a fixed version to upgrade to.

In [ ]:
%%writefile requirements_demo.txt
fastapi
PyYAML==5.3.1

In [ ]:
# ===== Option A: Google Colab =====
!pip-audit --no-deps --disable-pip -r requirements_demo.txt

# ===== Option B: VS Code terminal =====
# pip-audit --no-deps --disable-pip -r requirements_demo.txt

## 10. NEW (v4): Vulnerability report v1
Turn the findings into rows of the report. Columns follow `shared/templates/Vulnerability_Report_Template.md`.

In [ ]:
FIXES = {"B105": "Read SECRET_KEY from the environment", "B602": "Pass a list and shell=False",
         "B324": "Use bcrypt for passwords (sha256 for non-security ids)", "B307": "Use ast.literal_eval or a parser",
         "B404": "Keep subprocess, but never with shell=True"}
rows = ["| ID | Finding | Tool | Severity | Affected file | Proposed fix | Fixed by | Verified | Evidence |",
        "|---|---|---|---|---|---|---|---|---|"]
for i, f_ in enumerate(findings, 1):
    rows.append(f"| V-{i:02d} | {f_['issue_text'][:45]} | bandit {f_['test_id']} | {f_['issue_severity'].title()} | "
                f"{f_['filename']}:{f_['line_number']} | {FIXES.get(f_['test_id'], 'Review')} | (human or Codex-assisted) | No | |")
report = "# Vulnerability report v1\n\n" + "\n".join(rows)
open("vulnerability_report_v1.md", "w").write(report)
print(report)

### TODO 2: Fix two findings in `ai_generated_helpers.py` (you may ask Codex), re-run the bandit cell, and update the Verified and Evidence columns.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)